In [46]:
import pandas as pd
import requests, json
import numpy as np
import time  
import pygsheets
from dotenv import load_dotenv
import os
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

In [47]:
pd.set_option('display.max_columns', None)  # Mostra todas as colunas
pd.set_option('display.width', None)        # Desativa quebra de linha automática
pd.set_option('display.expand_frame_repr', False)  # Mostra tudo em uma linha, se possível

In [48]:
load_dotenv()  # Carrega as variáveis do .env

token = os.getenv("BEARER_TOKEN")
url_leads = os.getenv("URL_LEADS")
url_pipeline = os.getenv("URL_PIPELINE")

In [49]:
headers = {
    "accept": "application/json",
    "authorization": f"Bearer {token}"
}

In [50]:
response = requests.get(url_pipeline, headers=headers)

print(response.text)

{"_total_items":2,"_links":{"self":{"href":"https://atendimentocasadovolante.kommo.com/api/v4/leads/pipelines"}},"_embedded":{"pipelines":[{"id":11142155,"name":"Funil de vendas","sort":1,"is_main":true,"is_unsorted_on":true,"is_archive":false,"account_id":34589203,"_links":{"self":{"href":"https://atendimentocasadovolante.kommo.com/api/v4/leads/pipelines/11142155"}},"_embedded":{"statuses":[{"id":85491743,"name":"Leads de entrada","sort":10,"is_editable":false,"pipeline_id":11142155,"color":"#c1c1c1","type":1,"account_id":34589203,"_links":{"self":{"href":"https://atendimentocasadovolante.kommo.com/api/v4/leads/pipelines/11142155/statuses/85491743"}}},{"id":86825243,"name":"Entrada de leads fora de horário","sort":20,"is_editable":true,"pipeline_id":11142155,"color":"#99ccff","type":0,"account_id":34589203,"_links":{"self":{"href":"https://atendimentocasadovolante.kommo.com/api/v4/leads/pipelines/11142155/statuses/86825243"}}},{"id":85595887,"name":"REALIZAR CONTATO","sort":30,"is_edi

In [51]:
dados = response.json()['_embedded']['pipelines']

df1 = pd.DataFrame(dados)
df1.rename(columns={
    'id': 'id_pipeline',
    'name': 'pipeline'
}, inplace=True)
df1.head()

,id_pipeline,pipeline,sort,is_main,is_unsorted_on,is_archive,account_id,_links,_embedded
0,11142155,Funil de vendas,1,True,True,False,34589203,{'self': {'href': 'https://atendimentocasadovo...,"{'statuses': [{'id': 85491743, 'name': 'Leads ..."
1,11155003,Funil Follow-up (SEM RESPOSTA),2,False,True,False,34589203,{'self': {'href': 'https://atendimentocasadovo...,"{'statuses': [{'id': 85594935, 'name': 'Etapa ..."


In [52]:
# Cria nova coluna com os statuses
df1['statuses'] = df1['_embedded'].apply(lambda x: x.get('statuses', []))

# Expande a lista de statuses em linhas
df_statuses = df1.explode('statuses').reset_index(drop=True)

# Transforma os dicionários da coluna 'statuses' em colunas
df_statuses = pd.concat([df_statuses.drop(columns=['statuses', '_embedded']), df_statuses['statuses'].apply(pd.Series)], axis=1)


In [53]:
df_statuses = df_statuses[['id','name','pipeline']].drop_duplicates(subset='id')
df_statuses.rename(columns={
"name": "step_name"
}, inplace=True)

In [54]:
df_statuses

,id,step_name,pipeline
0,85491743,Leads de entrada,Funil de vendas
1,86825243,Entrada de leads fora de horário,Funil de vendas
2,85595887,REALIZAR CONTATO,Funil de vendas
3,85594383,CLIENTE RECORRENTE,Funil de vendas
4,85667671,REPAROS,Funil de vendas
5,85491991,CONTATO REALIZADO,Funil de vendas
6,85491995,LEAD POTENCIAL,Funil de vendas
7,85491999,ORÇAMENTO ENVIADO,Funil de vendas
8,85594019,AGENDADO O DIA,Funil de vendas
9,142,Venda Concluída 🤑,Funil de vendas


In [55]:
# Arquivo para salvar o timestamp da última execução
LAST_RUN_FILE = "last_run.json"

# Função para carregar o último timestamp
def carregar_ultimo_timestamp():
    if os.path.exists(LAST_RUN_FILE):
        try:
            with open(LAST_RUN_FILE, "r") as f:
                conteudo = f.read().strip()
                if not conteudo:
                    return None  # Arquivo existe mas está vazio
                dados = json.loads(conteudo)
                return dados.get("last_run", None)
        except json.JSONDecodeError:
            print("Erro ao ler JSON: conteúdo inválido. Ignorando arquivo.")
            return None
    return None


# Função para salvar novo timestamp
def salvar_timestamp(timestamp):
    with open(LAST_RUN_FILE, "w") as f:
        json.dump({"last_run": timestamp}, f)


In [56]:
# Recuperar o timestamp anterior ou iniciar com 24h atrás
timestamp_ultimo = carregar_ultimo_timestamp()
if timestamp_ultimo is None:
    print("Nenhuma execução anterior encontrada. Salvando timestamp inicial.")
    timestamp_ultimo = int((datetime.utcnow() - timedelta(days=1)).timestamp())
    salvar_timestamp(timestamp_ultimo)

In [57]:
# Fazer a chamada à API
all_leads = []
limit = 250
page = 1

while True:
    params = {
        "limit": limit,
        "page": page,
        "filter[created_at][from]": timestamp_ultimo
    }

    response = requests.get(url_leads, headers=headers, params=params)
    
    if response.status_code != 200:
        print(f"Erro na página {page}: {response.status_code}")
        break

    data = response.json().get('_embedded', {}).get('leads', [])
    
    if not data:
        break

    all_leads.extend(data)
    print(f"Página {page} carregada: {len(data)} registros")

    if len(data) < limit:
        break

    page += 1
    time.sleep(0.3)

print(f"\nTotal de leads novos coletados: {len(all_leads)}")

# Atualizar timestamp com a hora atual (fim da execução)
novo_timestamp = int(datetime.utcnow().timestamp())
salvar_timestamp(novo_timestamp)


Erro na página 1: 204

Total de leads novos coletados: 0


In [58]:
df = pd.DataFrame(all_leads)
df.head()

""


In [59]:
# -------- 1. Expandir custom_fields_values --------
def extract_custom_fields(row):
    if isinstance(row, list):
        output = {}
        for item in row:
            field_name = item.get('field_name')
            values = item.get('values', [])
            if field_name and values:
                output[field_name] = values[0].get('value')
        return pd.Series(output)
    return pd.Series()

custom_fields_df = df['custom_fields_values'].apply(extract_custom_fields)

# Opcional: limpar nomes de colunas (substituir '/' por '_', etc.)
custom_fields_df.columns = [col.replace('/', '_').replace(' ', '_') for col in custom_fields_df.columns]

# -------- 2. Extrair nomes das tags --------
def extract_tags(row):
    if isinstance(row, list):
        return [tag.get('name') for tag in row if isinstance(tag, dict)]
    return []

df['tags'] = df['_embedded'].apply(extract_tags)

# # -------- 3. Extrair nome da empresa --------
# def extract_company_name(row):
#     if isinstance(row, list) and len(row) > 0:
#         return row[0].get('name')
#     return None

# df['company_name'] = df['_embedded.companies'].apply(extract_company_name)

# -------- 4. Concatenar tudo no DataFrame final --------
df = pd.concat([df, custom_fields_df], axis=1)

# (Opcional) Remover as colunas originais se desejar
#df.drop(columns=['custom_fields_values', '_embedded.tags', '_embedded.companies'], inplace=True)



KeyError: 'custom_fields_values'

In [ ]:
df.head(5)

In [ ]:
df[df['name'] == "George"]

In [ ]:
#df.to_csv('df_kommo_v3.csv', encoding="utf-8-sig")

In [ ]:
print(type(df['created_at'][0]))
print(type(df['updated_at'][0]))
print(type(df['closed_at'][0]))
print(type(df['closest_task_at'][0]))
print(type(df['Data_do_agendamento'].iloc[0]))
print(type(df['Data_de_criação'].iloc[0]))

Exploração


In [ ]:
""" df[['created_at', 
	   'updated_at', 
	   'closed_at', 
	   'closest_task_at',
       'Data_do_agendamento', 
       'Data_de_criação']] """

In [ ]:
# Lista das colunas com Unix Timestamps
unix_columns = [
    'created_at', 
    'updated_at', 
    'closed_at', 
    'closest_task_at',
    'Data_do_agendamento', 
    'Data_de_criação'
]

# Limite superior para timestamps válidos (01/01/2100)
max_valid_timestamp = 4102444800  # segundos desde 1970

# Conversão segura
for col in unix_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')  # garante tipo numérico
        df[col] = df[col].where(df[col] < max_valid_timestamp, np.nan)  # remove absurdos
        df[col] = pd.to_datetime(df[col], unit='s', errors='coerce')  # converte para datetime


In [ ]:
df[['created_at', 
	   'updated_at', 
	   'closed_at', 
	   'closest_task_at',
       'Data_do_agendamento', 
       'Data_de_criação']]

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df_statuses.head()

In [ ]:
# Renomear colunas do mapeamento para evitar conflito
df_statuses = df_statuses.rename(columns={
    'id': 'status_id',       # para bater com o df_principal
    'step_name': 'status_name',
    'pipeline': 'pipeline_name'
})
df['status_id'] = df['status_id'].astype('Int64')  # Int64 com I maiúsculo permite valores NaN
df_statuses['status_id'] = df_statuses['status_id'].astype('Int64')

# Realizar o merge com base no status_id
df = df.merge(df_statuses, on='status_id', how='left')

In [ ]:
# Mapeamento dos IDs para nomes
mapa_vendedores = {
    "11188591": "Everton Oliveira",
    "13190131": "Vendedora Gabriele",
    "13190615": "Vendedor Daniel",
    "13190631": "Vendedor Leonardo"
}

# Converta a coluna responsible_user_id para string (se ainda não for)
df['responsible_user_id'] = df['responsible_user_id'].astype(str)

# Crie ou atualize a coluna 'vendedor' com base no dicionário
df['vendedor'] = df['responsible_user_id'].map(mapa_vendedores)


In [ ]:
#Tempo médio entre criação e agendamento
df['tempo_ate_agendamento'] = (df['Data_do_agendamento'] - df['Data_de_criação']).dt.days
#ciclo de venda (em dias)
df['ciclo_venda_dias'] = (df['closed_at'] - df['created_at']).dt.days


In [ ]:
df['agendou'] = df['Data_do_agendamento'].notna()


In [ ]:

df[df['status_name'] == 'Venda Concluída 🤑']


In [ ]:
df[['created_at','closed_at','ciclo_venda_dias']].head(10)

In [ ]:
df[df['id'] == "3192024"]

In [ ]:
import pygsheets

# Autenticar com a API do Google Sheets
gc = pygsheets.authorize(service_file='credenciais.json')

# Abrir planilha e worksheet
sh = gc.open('Dados Vendas Casa do Volante')
wks = sh.worksheet_by_title('kommo-api')

# ✅ Carregar os IDs já existentes
existing_data = wks.get_all_records()
existing_ids = {str(row['id']) for row in existing_data if 'id' in row}


# ✅ Garantir que 'id' esteja como string para comparação
df['id'] = df['id'].astype(str)

# ✅ Filtrar apenas registros com IDs que ainda não estão na planilha
df_novos = df[~df['id'].isin(existing_ids)]

print(f"Leads novos a adicionar: {len(df_novos)}")


In [ ]:

# ✅ Append apenas se houver dados novos
if not df_novos.empty:
    next_row = len(wks.get_all_values(include_tailing_empty=False)) + 1
    wks.set_dataframe(df_novos, (next_row, 1))
    print("Novos leads adicionados ao Google Sheets.")
else:
    print("Nenhum lead novo para adicionar.")

